In [1]:
using JLD2
using StaticArrays
using WaveGreen2D.Chebyshev: ChebyshevSeries, TransformedChebyshevSeries, gradient, hessian, contains

In [2]:
function u(x::SVector{3, Float64})
    return SVector{3, Float64}(x[1], x[2], log(x[3]))
end


function ∇u(x::SVector{3, Float64})
    grad_u = one(MMatrix{3, 3, Float64, 9})
    grad_u[3, 3] = 1/x[3]
    return SMatrix{3, 3, Float64, 9}(grad_u)
end


function Hu(x::SVector{3, Float64})
    hess_u = zero(MArray{Tuple{3, 3, 3}, Float64, 3, 27})
    hess_u[3, 3, 3] = -1/x[3]^2
    return SArray{Tuple{3, 3, 3}, Float64, 3, 27}(hess_u)
end


cs = jldopen("../src/chebyshev_series.jld2")

const L₁_series = read(cs, "L₁_series")
const L₂_series = read(cs, "L₂_series")

close(cs);

In [9]:
function get_L₁_series_index(x::SVector{3, Float64})
    u = SVector{3, Float64}(x[1], x[2], log(x[3]))
    
    if contains(L₁_series[1], x)
        return 1
    elseif contains(L₁_series[2], u)
        return 2
    elseif contains(L₁_series[3], u)
        return 3
    else
        throw(DomainError(x, "Point not in the domain of L₁ Chebyshev series."))
    end
end


function get_L₂_series_index(x::SVector{3, Float64})
    u = SVector{3, Float64}(x[1], x[2], log(x[3]))
    
    if contains(L₂_series[1], u)
        return 1
    elseif contains(L₂_series[2], u)
        return 2
    elseif contains(L₂_series[3], u)
        return 3
    elseif contains(L₂_series[4], u)
        return 4
    else
        throw(DomainError(x, "Point not in the domain of L₂ Chebyshev series."))
    end
end


function L₁(A::Float64, B₁::Float64, H::Float64)
    x = SVector{3, Float64}(A, B₁, H)
    u = SVector{3, Float64}(A, B₁, log(H))
    
    i = get_L₁_series_index(x)

    if i == 1
        return L₁_series[i](x)
    end
    
    return L₁_series[i](u)
end


function ∇L₁(A::Float64, B₁::Float64, H::Float64)
    x = SVector{3, Float64}(A, B₁, H)
    u = SVector{3, Float64}(A, B₁, log(H))
    
    i = get_L₁_series_index(x)

    if i == 1
        return gradient(L₁_series[i], x)
    end
    
    L, ∇ᵤL = gradient(L₁_series[i], u)
    
    # ∂L/∂x = ∂L/∂u ⋅ ∂u/∂x
    ∇L = SVector{3, Float64}(∇ᵤL[1], ∇ᵤL[2], ∇ᵤL[3]/H)
    
    return L, ∇L
end


function HL₁(A::Float64, B₁::Float64, H::Float64)
    x = SVector{3, Float64}(A, B₁, H)
    u = SVector{3, Float64}(A, B₁, log(H))
    
    i = get_L₁_series_index(x)

    if i == 1
        return hessian(L₁_series[i], x)
    end
        
    L, ∇ᵤL, HᵤL = hessian(L₁_series[i], u)
    
    # ∂L/∂x = ∂L/∂u ⋅ ∂u/∂x
    ∇L = SVector{3, Float64}(∇ᵤL[1], ∇ᵤL[2], ∇ᵤL[3]/H)

    # ∂²L/∂x² = ∂L/∂u ⋅ ∂²u/∂x² + ∂²L/∂u² ⋅ (∂u/∂x)²
    HL¹ = zero(MMatrix{3, 3, Float64, 9})
    HL¹[3, 3] = -∇ᵤL[3] / H^2

    HL² = MMatrix{3, 3, Float64, 9}(HᵤL)
    HL²[3, :] .*= 1/H
    HL²[:, 3] .*= 1/H
    
    HL = SMatrix{3, 3, Float64, 9}(HL¹ + HL²)

    return L, ∇L, HL
end


function L₂(A::Float64, B₂::Float64, H::Float64)
    x = SVector{3, Float64}(A, B₂, H)
    u = SVector{3, Float64}(A, B₂, log(H))
    i = get_L₂_series_index(x)
    L = L₂_series[i](u)
    return L
end


# function ∇L₂(A::Float64, B₂::Float64, H::Float64)
#     x = SVector{3, Float64}(A, B₂, H)

#     if contains(L₂1, x)
#         return gradient(L₂1, x)
#     elseif contains(L₂2, x)
#         return gradient(L₂2, x)
#     elseif contains(L₂3, x)
#         return gradient(L₂3, x)
#     else
#         throw(DomainError(x, L₂_domain_error_msg))
#     end
# end


# function HL₂(A::Float64, B₂::Float64, H::Float64)
#     x = SVector{3, Float64}(A, B₂, H)

#     if contains(L₂1, x)
#         return hessian(L₂1, x)
#     elseif contains(L₂2, x)
#         return hessian(L₂2, x)
#     elseif contains(L₂3, x)
#         return hessian(L₂3, x)
#     else
#         throw(DomainError(x, L₂_domain_error_msg))
#     end
# end;
;

$L_1$

-1.00169204874032

[-0.06799288769514582, 0.28012825734268715, 0.15870001060024233]

[-0.6732390207590058 -0.07372986793058411 0.08936761425251266; -0.07372986793058411 0.6732390207590058 -0.5061078894933684; 0.08936761425251266 -0.5061078894933684 0.11516526881129213]

In [4]:
L₁(0.1, 0.5, 0.3)

-1.0016920487403178

In [5]:
L, ∇L = ∇L₁(0.1, 0.5, 0.3)
println(L)
println(∇L)

-1.0016920487403178
[-0.0679928876951603, 0.2801282573427111, 0.15870001060026287]


In [6]:
L, ∇L, HL = HL₁(0.1, 0.5, 0.3)
println(L)
println(∇L)
println(HL)

-1.0016920487403178
[-0.0679928876951603, 0.2801282573427111, 0.15870001060026287]
[-0.6732390207596641 -0.07372986793040157 0.08936761425249547; -0.07372986793040157 0.6732390207586858 -0.5061078894934178; 0.08936761425249547 -0.5061078894934178 0.11516526881072357]


In [7]:
HL

3×3 SMatrix{3, 3, Float64, 9} with indices SOneTo(3)×SOneTo(3):
 -0.673239   -0.0737299   0.0893676
 -0.0737299   0.673239   -0.506108
  0.0893676  -0.506108    0.115165

In [23]:
a = rand(MMatrix{3, 3, Float64, 9})
b = rand(MMatrix{3, 3, Float64, 9})
c = a + b
typeof(c) <: MMatrix  # returns true
typeof(c) <: SMatrix  # returns false
# a[1, 1] = 1.0
# a[1, :] *= 3.0
# a + b

false

In [30]:
H = 1.0

∇ᵤL = rand(SVector{3, Float64})
HᵤL = rand(SMatrix{3, 3, Float64, 9})

HL¹ = zero(MMatrix{3, 3, Float64, 9})
HL¹[3, 3] = -∇ᵤL[3] / H^2

HL² = MMatrix{3, 3, Float64, 9}(HᵤL)
HL²[3, :] .*= 1/H
HL²[:, 3] .*= 1/H

HL = HL¹ + HL²
# HL = SMatrix{3, 3, Float64, 9}(HL¹ + HL²)

3×3 MMatrix{3, 3, Float64, 9} with indices SOneTo(3)×SOneTo(3):
 0.0905451  0.896655   0.499674
 0.884426   0.52489    0.807986
 0.37832    0.381869  -0.0215996

$L_2$

-1.039841625349199

[0.012907603643945642, -0.2727287078066397, 6.283285000126575]

[0.13029525572290562 0.022550617007363834 0.1454782528537148; 0.022550617007363834 -0.13029525572290554 1.700800437397748; 0.1454782528537148 1.700800437397748 -31.315956117865]

In [10]:
@code_warntype L₁(0.1, 0.5, 0.3)

MethodInstance for L₁(::Float64, ::Float64, ::Float64)
  from L₁(A::Float64, B₁::Float64, H::Float64) @ Main In[3]:33
Arguments
  #self#::Core.Const(Main.L₁)
  A::Float64
  B₁::Float64
  H::Float64
Locals
  i::Int64
  u::SVector{3, Float64}
  x::SVector{3, Float64}
Body::Float64
1 ─ %1  = Main.SVector::Core.Const(SVector)
│   %2  = Main.Float64::Core.Const(Float64)
│   %3  = Core.apply_type(%1, 3, %2)::Core.Const(SVector{3, Float64})
│         (x = (%3)(A, B₁, H))
│   %5  = Main.SVector::Core.Const(SVector)
│   %6  = Main.Float64::Core.Const(Float64)
│   %7  = Core.apply_type(%5, 3, %6)::Core.Const(SVector{3, Float64})
│   %8  = Main.log::Core.Const(log)
│   %9  = (%8)(H)::Float64
│         (u = (%7)(A, B₁, %9))
│   %11 = Main.get_L₁_series_index::Core.Const(Main.get_L₁_series_index)
│   %12 = x::SVector{3, Float64}
│         (i = (%11)(%12))
│   %14 = Main.:(==)::Core.Const(==)
│   %15 = i::Int64
│   %16 = (%14)(%15, 1)::Bool
└──       goto #3 if not %16
2 ─ %18 = Main.L₁_series::Core

In [11]:
@code_warntype ∇L₁(0.1, 0.5, 0.3)

MethodInstance for ∇L₁(::Float64, ::Float64, ::Float64)
  from ∇L₁(A::Float64, B₁::Float64, H::Float64) @ Main In[3]:47
Arguments
  #self#::Core.Const(Main.∇L₁)
  A::Float64
  B₁::Float64
  H::Float64
Locals
  @_5::Int64
  ∇L::SVector{3, Float64}
  ∇ᵤL::SVector{3, Float64}
  L::Float64
  i::Int64
  u::SVector{3, Float64}
  x::SVector{3, Float64}
Body::Tuple{Float64, SVector{3, Float64}}
1 ─       Core.NewvarNode(:(@_5))
│         Core.NewvarNode(:(∇L))
│         Core.NewvarNode(:(∇ᵤL))
│         Core.NewvarNode(:(L))
│   %5  = Main.SVector::Core.Const(SVector)
│   %6  = Main.Float64::Core.Const(Float64)
│   %7  = Core.apply_type(%5, 3, %6)::Core.Const(SVector{3, Float64})
│         (x = (%7)(A, B₁, H))
│   %9  = Main.SVector::Core.Const(SVector)
│   %10 = Main.Float64::Core.Const(Float64)
│   %11 = Core.apply_type(%9, 3, %10)::Core.Const(SVector{3, Float64})
│   %12 = Main.log::Core.Const(log)
│   %13 = (%12)(H)::Float64
│         (u = (%11)(A, B₁, %13))
│   %15 = Main.get_L₁_series_in

In [10]:
@code_warntype HL₁(0.1, 0.5, 0.3)

MethodInstance for HL₁(::Float64, ::Float64, ::Float64)
  from HL₁(A::Float64, B₁::Float64, H::Float64) @ Main In[9]:66
Arguments
  #self#::Core.Const(Main.HL₁)
  A::Float64
  B₁::Float64
  H::Float64
Locals
  @_5::Int64
  HL::SMatrix{3, 3, Float64, 9}
  HL²::MMatrix{3, 3, Float64, 9}
  HL¹::MMatrix{3, 3, Float64, 9}
  ∇L::SVector{3, Float64}
  HᵤL::SMatrix{3, 3, Float64, 9}
  ∇ᵤL::SVector{3, Float64}
  L::Float64
  i::Int64
  u::SVector{3, Float64}
  x::SVector{3, Float64}
Body::Tuple{Float64, SVector{3, Float64}, SMatrix{3, 3, Float64, 9}}
1 ─        Core.NewvarNode(:(@_5))
│          Core.NewvarNode(:(HL))
│          Core.NewvarNode(:(HL²))
│          Core.NewvarNode(:(HL¹))
│          Core.NewvarNode(:(∇L))
│          Core.NewvarNode(:(HᵤL))
│          Core.NewvarNode(:(∇ᵤL))
│          Core.NewvarNode(:(L))
│   %9   = Main.SVector::Core.Const(SVector)
│   %10  = Main.Float64::Core.Const(Float64)
│   %11  = Core.apply_type(%9, 3, %10)::Core.Const(SVector{3, Float64})
│          (x 

In [8]:
@code_warntype HL₁(0.1, 0.5, 0.3)

MethodInstance for HL₁(::Float64, ::Float64, ::Float64)
  from HL₁(A::Float64, B₁::Float64, H::Float64) @ Main In[3]:66
Arguments
  #self#::Core.Const(Main.HL₁)
  A::Float64
  B₁::Float64
  H::Float64
Locals
  @_5::Int64
  HL::MMatrix{3, 3, Float64, 9}
  HL²::MMatrix{3, 3, Float64, 9}
  HL¹::MMatrix{3, 3, Float64, 9}
  ∇L::SVector{3, Float64}
  HᵤL::SMatrix{3, 3, Float64, 9}
  ∇ᵤL::SVector{3, Float64}
  L::Float64
  i::Int64
  u::SVector{3, Float64}
  x::SVector{3, Float64}
Body::Union{Tuple{Float64, SVector{3, Float64}, SMatrix{3, 3, Float64, 9}}, Tuple{Float64, SVector{3, Float64}, MMatrix{3, 3, Float64, 9}}}
1 ─        Core.NewvarNode(:(@_5))
│          Core.NewvarNode(:(HL))
│          Core.NewvarNode(:(HL²))
│          Core.NewvarNode(:(HL¹))
│          Core.NewvarNode(:(∇L))
│          Core.NewvarNode(:(HᵤL))
│          Core.NewvarNode(:(∇ᵤL))
│          Core.NewvarNode(:(L))
│   %9   = Main.SVector::Core.Const(SVector)
│   %10  = Main.Float64::Core.Const(Float64)
│   %11  = Core

In [6]:
@code_warntype L₂(0.2, 1.0, 3.0)

MethodInstance for L₂(::Float64, ::Float64, ::Float64)
  from L₂(A::Float64, B₂::Float64, H::Float64) @ Main In[3]:84
Arguments
  #self#::Core.Const(Main.L₂)
  A::Float64
  B₂::Float64
  H::Float64
Locals
  x::SVector{3, Float64}
Body::Float64
1 ─ %1  = Main.SVector::Core.Const(SVector)
│   %2  = Main.Float64::Core.Const(Float64)
│   %3  = Core.apply_type(%1, 3, %2)::Core.Const(SVector{3, Float64})
│         (x = (%3)(A, B₂, H))
│   %5  = Main.contains::Core.Const(WaveGreen2D.Chebyshev.contains)
│   %6  = Main.L₂1::Core.Const(3-D transformed Chebyshev series of order (11, 18, 23))
│   %7  = x::SVector{3, Float64}
│   %8  = (%5)(%6, %7)::Bool
└──       goto #3 if not %8
2 ─ %10 = Main.L₂1::Core.Const(3-D transformed Chebyshev series of order (11, 18, 23))
│   %11 = x::SVector{3, Float64}
│   %12 = (%10)(%11)::Float64
└──       return %12
3 ─ %14 = Main.contains::Core.Const(WaveGreen2D.Chebyshev.contains)
│   %15 = Main.L₂2::Core.Const(3-D transformed Chebyshev series of order (11, 18, 2

In [4]:


function C₁(x::SVector{3, Float64})
    if contains(C₁1, x)
        return C₁1(x)
    elseif contains(C₁2, x)
        return C₁2(x)
    elseif contains(C₁3, x)
        return C₁3(x)
    else
        throw(DomainError(x))
    end
end


function C₁_gradient(x::SVector{3, Float64})
    if contains(C₁1, x)
        return gradient(C₁1, x)
    elseif contains(C₁2, x)
        return gradient(C₁2, x)
    elseif contains(C₁3, x)
        return gradient(C₁3, x)
    else
        throw(DomainError(x))
    end
end


function C₁_hessian(x::SVector{3, Float64})
    if contains(C₁1, x)
        return hessian(C₁1, x)
    elseif contains(C₁2, x)
        return hessian(C₁2, x)
    elseif contains(C₁3, x)
        return hessian(C₁3, x)
    else
        throw(DomainError(x))
    end
end


function C₂(x::SVector{3, Float64})
    if contains(C₂1, x)
        return C₂1(x)
    elseif contains(C₂2, x)
        return C₂2(x)
    elseif contains(C₂3, x)
        return C₂3(x)
    elseif contains(C₂4, x)
        return C₂4(x)
    else
        throw(DomainError(x))
    end
end


function C₂_gradient(x::SVector{3, Float64})
    if contains(C₂1, x)
        return gradient(C₂1, x)
    elseif contains(C₂2, x)
        return gradient(C₂2, x)
    elseif contains(C₂3, x)
        return gradient(C₂3, x)
    elseif contains(C₂4, x)
        return gradient(C₂4, x)
    else
        throw(DomainError(x))
    end
end


function C₂_hessian(x::SVector{3, Float64})
    if contains(C₂1, x)
        return hessian(C₂1, x)
    elseif contains(C₂2, x)
        return hessian(C₂2, x)
    elseif contains(C₂3, x)
        return hessian(C₂3, x)
    elseif contains(C₂4, x)
        return hessian(C₂4, x)
    else
        throw(DomainError(x))
    end
end;

In [6]:
tol_L = 10^2 * eps()
tol_∇L = 10^4 * eps()
tol_HL = 10^6 * eps();

In [7]:
@testset "L₁ Chebyshev" begin
    @load "../test/data/l1_test_data.jld2" x1 L1 ∇L1 HL1

    npoints = length(x1)
    
    C1 = Vector{Float64}(undef, npoints)
    C1g = Vector{Float64}(undef, npoints)
    C1h = Vector{Float64}(undef, npoints)
    
    ∇C1 = Matrix{Float64}(undef, npoints, 3)
    ∇C1h = Matrix{Float64}(undef, npoints, 3)
    
    HC1 = Array{Float64, 3}(undef, npoints, 3, 3)

    for i in 1:npoints
        y₁ = C₁(x1[i])
        y₂, ∇y₂ = C₁_gradient(x1[i])
        y₃, ∇y₃, Hy₃ = C₁_hessian(x1[i])
    
        C1[i] = y₁
        C1g[i] = y₂
        C1h[i] = y₃
        ∇C1[i, :] .= ∇y₂
        ∇C1h[i, :] .= ∇y₃
        HC1[i, :, :] .= Hy₃
    end

    # Test if ChebyshevSeries, gradient and hessian return the same value of L₁
    @test C1 == C1g
    @test C1 == C1h

    # Test if gradient and hessian return the same value of ∇L₁
    @test ∇C1 == ∇C1h

    @test all(isapprox.(L1, C1; rtol=tol_L, atol=tol_L))
    @test all(isapprox.(∇L1, ∇C1; rtol=tol_∇L, atol=tol_∇L))
    @test all(isapprox.(HL1, HC1; rtol=tol_HL, atol=tol_HL))
end

Test Summary: | Pass  Total  Time
L₁ Chebyshev  |    6      6  8.4s


Test.DefaultTestSet("L₁ Chebyshev", Any[], 6, false, false, true, 1.782325593430538e9, 1.782325601793092e9, false, "In[7]", Random.Xoshiro(0xcf472b10b95f8e61, 0x89133bd8ee98d2f4, 0x9f1f89be3af83969, 0x6217d4e58df07103, 0x34c561998d963d5f))

In [8]:
@testset "L₂ Chebyshev" begin
    @load "../test/data/l2_test_data.jld2" x2 L2 ∇L2 HL2

    npoints = length(x2)
    
    C2 = Vector{Float64}(undef, npoints)
    C2g = Vector{Float64}(undef, npoints)
    C2h = Vector{Float64}(undef, npoints)
    
    ∇C2 = Matrix{Float64}(undef, npoints, 3)
    ∇C2h = Matrix{Float64}(undef, npoints, 3)
    
    HC2 = Array{Float64, 3}(undef, npoints, 3, 3)

    for i in 1:npoints
        y₁ = C₂(x2[i])
        y₂, ∇y₂ = C₂_gradient(x2[i])
        y₃, ∇y₃, Hy₃ = C₂_hessian(x2[i])
    
        C2[i] = y₁
        C2g[i] = y₂
        C2h[i] = y₃
        ∇C2[i, :] .= ∇y₂
        ∇C2h[i, :] .= ∇y₃
        HC2[i, :, :] .= Hy₃
    end

    # Test if ChebyshevSeries, gradient and hessian return the same value of L₁
    @test C2 == C2g
    @test C2 == C2h

    # Test if gradient and hessian return the same value of ∇L₁
    @test ∇C2 == ∇C2h

    @test all(isapprox.(L2, C2; rtol=tol_L, atol=tol_L))
    @test all(isapprox.(∇L2, ∇C2; rtol=tol_∇L, atol=tol_∇L))
    @test all(isapprox.(HL2, HC2; rtol=tol_HL, atol=tol_HL))
end

Test Summary: | Pass  Total  Time
L₂ Chebyshev  |    6      6  0.2s


Test.DefaultTestSet("L₂ Chebyshev", Any[], 6, false, false, true, 1.782325602176509e9, 1.782325602334543e9, false, "In[8]", Random.Xoshiro(0xcf472b10b95f8e61, 0x89133bd8ee98d2f4, 0x9f1f89be3af83969, 0x6217d4e58df07103, 0x34c561998d963d5f))